# Живое демо: Repo Analyzer → Candidate Profile Builder (Qwen3.5-9B Q4_K_M)

Этот ноутбук проверяет **PR #30**: `GitHub → Repo Analyzer → analysis.json → LLM Candidate Profile Builder → навыки и абзацы проектов`. Новая ML-стадия получает уже готовый анализ и не скачивает исходники из GitHub.

**Запуск:** Google Colab → *Runtime → Change runtime type → T4 GPU* → выполняй ячейки сверху вниз. Нужны доступ в интернет и место под GGUF (~5 GB). Применяется код из ветки `feat/candidate-profile-builder`. Для наблюдения работы LLM сохраняются её фактические предложения навыков и описаний.

**Важно:** это smoke test на реальной модели, **не** проверка semantic grounding и не benchmark качества. Успешное завершение CLI (код 0) само по себе не означает, что агент нашёл хорошие факты: проверь `status`, `tool_trace`, `evidence` и доказательства по ссылкам. Не загружай приватные репозитории без отдельной проверки безопасности исходников.

## 1. Настройки и GPU

Для первого запуска оставь публичный `military-charters-rag`. Модель и сервер находятся **в этом Colab runtime**, без удалённого LLM API. При необходимости укажи `GITHUB_TOKEN` в Colab Secrets; токен не требуется для одного небольшого публичного репозитория.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

TARGET_REPO = "Ferraronp/military-charters-rag"  # @param {type:"string"}
MAX_STEPS = 10  # @param {type:"integer"}
MAX_READS = 4  # @param {type:"integer"}
MODEL_TEMPERATURE = 0.5  # @param {type:"number"}  (было 0.1; для A/B можно вернуть обратно)
CANDIDATE_LOGIN = "demo_developer"  # @param {type:"string"}
# Можно добавить пути к ранее сохранённым analysis.json / smoke ZIP других репозиториев.
CACHED_ANALYSIS_FILES = []

PR_BRANCH = "feat/candidate-profile-builder"
REPO_URL = "https://github.com/Ferraronp/github2CV_ML.git"
REPO_DIR = Path("/content/github2cv_profile_pr30")
WORKDIR = Path("/content/github2cv_profile_demo")
WORKDIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR = WORKDIR / "results"

MODEL_REPO = "unsloth/Qwen3.5-9B-GGUF"
MODEL_FILE = "Qwen3.5-9B-Q4_K_M.gguf"
MODEL_ALIAS = "qwen-3.5"

assert 1 <= MAX_STEPS <= 50
assert 1 <= MAX_READS <= MAX_STEPS
assert 0 <= MODEL_TEMPERATURE <= 2
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
     "--format=csv,noheader"],
    check=True, capture_output=True, text=True,
)
try:
    from google.colab import userdata
except ImportError:
    userdata = None
if userdata is not None:
    try:
        github_token = userdata.get("GITHUB_TOKEN")
    except Exception:
        github_token = None
    if github_token:
        os.environ["GITHUB_TOKEN"] = github_token
print("GPU:", gpu.stdout.strip())
print("Repo to inspect:", TARGET_REPO)
print("Explorer budget:", MAX_STEPS, "steps /", MAX_READS, "reads")
print("Model:", MODEL_REPO, MODEL_FILE, "| temperature:", MODEL_TEMPERATURE)

## 2. Установка текущей ветки MVP и Python-зависимостей

Здесь не выполняется архивный Colab v8, а устанавливается **именно приложение из PR #30**.

In [ ]:
if REPO_DIR.exists():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--single-branch",
         "--branch", PR_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

active_branch = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True
).strip()
assert active_branch == PR_BRANCH, f"Expected {PR_BRANCH}, got {active_branch}"
commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
print("Agent source commit:", commit)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"],
    check=True,
)
print("MVP package and Hugging Face client installed.")

## 3. Готовый официальный CUDA-бинарник `llama-server` — без сборки

Используем **ровно способ из приложенного `Aiva_Character998_WebUI_AB.ipynb`**:
официальный GitHub Release `ggml-org/llama.cpp`, тег `b11896`, готовый архив
`llama-b11896-bin-ubuntu-cuda-12.8-x64.tar.gz`.
Проверяем SHA-256 перед распаковкой и повторно используем уже скачанный архив.
Если системных CUDA-библиотек не хватает, подтягиваем второй готовый архив
`cudart-llama-b11896-bin-ubuntu-cuda-12.8-x64.tar.gz`, также с проверкой SHA-256.

Никаких `cmake`, `make`, `nvcc` или сборки C++/CUDA из исходников.
Бинарник рассчитан на **актуальный Colab Ubuntu 24.04 x64 с NVIDIA T4**.
Проверяем `llama-server --version`, динамические зависимости через `ldd`
и наличие CUDA/T4 через `--list-devices`. Если окружение не подходит —
показываем конкретную ошибку, не подменяя CUDA на CPU.

Для нашего GitHub→CV проекта не требуются LoRA, Google Drive и Web UI из
ноутбука Айвы: перенесён только способ установки CUDA-сервера.

In [ ]:
import hashlib
import platform
import shutil
import tarfile
import urllib.error
import urllib.request
from pathlib import Path

RELEASE_TAG = "b11896"
RELEASE_COMMIT = "9125a2ebe8e5a38f0f4232170c134a29ed730b66"
RELEASE_BASE = (
    f"https://github.com/ggml-org/llama.cpp/releases/download/{RELEASE_TAG}/"
)
BIN_NAME = "llama-b11896-bin-ubuntu-cuda-12.8-x64.tar.gz"
BIN_SHA256 = "3e56cb1c4a31326695841d416933e4b404e05e3c00fd0efc81ec5cf07dc2c9b6"
CUDART_NAME = "cudart-llama-b11896-bin-ubuntu-cuda-12.8-x64.tar.gz"
CUDART_SHA256 = "d35d194f121a3905529877a1c8903db16d103245979f2d59ce14219611262a14"

assert platform.machine() in ("x86_64", "amd64"), "Нужен Linux x64"
glibc_text = platform.libc_ver()[1]
glibc_version = tuple(int(x) for x in glibc_text.split(".")[:2])
assert glibc_version >= (2, 39), (
    f"Официальному бинарнику нужен актуальный Colab (glibc >= 2.39); сейчас {glibc_text}"
)
device_info = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
    text=True,
).strip()
assert "T4" in device_info and "7.5" in device_info, (
    f"Этот pinned CUDA release проверяется на T4/SM75; обнаружено: {device_info}"
)

ARCHIVE_CACHE = WORKDIR / "official_release_archives"
SERVER_HOME = WORKDIR / f"llama_cpp_official_{RELEASE_TAG}"
RUNTIME_HOME = WORKDIR / f"llama_cpp_cudart_{RELEASE_TAG}"

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as source:
        for block in iter(lambda: source.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def download_verified(url, filename, expected_sha):
    dest = ARCHIVE_CACHE / filename
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.is_file() and sha256_file(dest) == expected_sha:
        print("Из кэша:", filename)
        return dest
    tmp = dest.with_suffix(dest.suffix + ".part")
    tmp.unlink(missing_ok=True)
    digest = hashlib.sha256()
    total = 0
    request = urllib.request.Request(
        url, headers={"User-Agent": "github2cv-colab-smoke"},
    )
    print("Скачиваем официальный релиз:", filename, flush=True)
    try:
        with urllib.request.urlopen(request, timeout=90) as response, tmp.open("wb") as out:
            for block in iter(lambda: response.read(8 * 1024 * 1024), b""):
                out.write(block)
                digest.update(block)
                total += len(block)
    except (urllib.error.URLError, TimeoutError, OSError) as exc:
        tmp.unlink(missing_ok=True)
        raise RuntimeError(
            f"Не удалось скачать {filename} из официального релиза: {exc}"
        ) from exc
    actual_sha = digest.hexdigest()
    if actual_sha != expected_sha:
        tmp.unlink(missing_ok=True)
        raise RuntimeError(
            f"SHA-256 не совпал для {filename}: {actual_sha} != {expected_sha}"
        )
    tmp.replace(dest)
    print(f"Скачано {total / 1024**2:.1f} MiB, SHA-256 совпал")
    return dest

def unpack_verified(archive, destination, expected_sha):
    marker = destination / ".archive_sha256"
    if marker.is_file() and marker.read_text() == expected_sha:
        return destination
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True)
    with tarfile.open(archive, "r:gz") as packed:
        packed.extractall(destination, filter="data")
    marker.write_text(expected_sha)
    return destination

binary_archive = download_verified(
    RELEASE_BASE + BIN_NAME, BIN_NAME, BIN_SHA256
)
binary_root = unpack_verified(binary_archive, SERVER_HOME, BIN_SHA256)
found = list(binary_root.rglob("llama-server"))
assert len(found) == 1, f"Ожидали один llama-server, обнаружено: {len(found)}"
SERVER_BIN = found[0]
SERVER_BIN.chmod(SERVER_BIN.stat().st_mode | 0o111)
cuda_backend = next(binary_root.rglob("libggml-cuda.so"), None)
assert cuda_backend is not None, "В архиве отсутствует CUDA backend"

def library_env(extra_dirs=()):
    env = os.environ.copy()
    binary_dirs = {str(SERVER_BIN.parent)}
    binary_dirs.update(str(path.parent) for path in binary_root.rglob("*.so*"))
    dirs = sorted(binary_dirs)
    dirs.extend(str(path) for path in extra_dirs)
    if env.get("LD_LIBRARY_PATH"):
        dirs.append(env["LD_LIBRARY_PATH"])
    env["LD_LIBRARY_PATH"] = os.pathsep.join(dirs)
    return env

SERVER_ENV = library_env()
linked = subprocess.run(
    ["ldd", str(cuda_backend)], env=SERVER_ENV,
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    check=True,
)
cuda_missing = any(
    f"{lib} => not found" in linked.stdout
    for lib in ("libcudart.so.12", "libcublas.so.12", "libcublasLt.so.12")
)
if cuda_missing:
    runtime_archive = download_verified(
        RELEASE_BASE + CUDART_NAME, CUDART_NAME, CUDART_SHA256
    )
    runtime_root = unpack_verified(
        runtime_archive, RUNTIME_HOME, CUDART_SHA256
    )
    runtime_dirs = sorted({path.parent for path in runtime_root.rglob("*.so*")})
    SERVER_ENV = library_env(runtime_dirs)

for binary in (SERVER_BIN, cuda_backend):
    link_check = subprocess.run(
        ["ldd", str(binary)], env=SERVER_ENV,
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=True,
    )
    unresolved = [
        row.strip() for row in link_check.stdout.splitlines()
        if "not found" in row
    ]
    assert not unresolved, (
        f"Отсутствуют библиотеки для {binary.name}: " + "; ".join(unresolved)
    )

version = subprocess.run(
    [str(SERVER_BIN), "--version"], env=SERVER_ENV,
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
assert version.returncode == 0, (
    f"Готовый llama-server не запускается: {version.stdout[-3000:]}"
)
assert RELEASE_COMMIT[:7] in version.stdout, (
    "Версия бинарника не совпала с закреплённой версией релиза: "
    + version.stdout[-2000:]
)
devices = subprocess.run(
    [str(SERVER_BIN), "--list-devices"], env=SERVER_ENV,
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
print("Версия:", version.stdout.strip())
print("Устройства:", devices.stdout.strip())
assert devices.returncode == 0 and "CUDA" in devices.stdout and "T4" in devices.stdout, (
    "Готовая сборка не видит CUDA/T4. Автоматический переход на CPU отключён."
)
print("Готовый официальный llama-server установлен — без компиляции:", SERVER_BIN)


## 4. Загрузка GGUF

Здесь загружается **Qwen3.5-9B Q4_K_M** из `unsloth/Qwen3.5-9B-GGUF` на Hugging Face. Это та же модель и квантование, которые мы уже проверяли в предыдущих smoke-прогонах. Режим reasoning при запуске `llama-server` отключён (`--reasoning off`), чтобы ответы оставались совместимыми с JSON/tool calling.

Температура для Explorer, Extractor и Profile Builder настраивается в первой ячейке (`MODEL_TEMPERATURE`, сейчас `0.5`). Она не гарантирует исправления некорректных `tool_calls`, поэтому диагностическая ячейка остаётся обязательной.

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

MODEL_DIR = WORKDIR / "models"
MODEL_DIR.mkdir(exist_ok=True)
model_info = HfApi().model_info(MODEL_REPO)
model_revision = model_info.sha
available_files = {file.rfilename for file in model_info.siblings}
assert MODEL_FILE in available_files, f"GGUF not available: {MODEL_FILE}"
MODEL_PATH = Path(hf_hub_download(
    repo_id=MODEL_REPO, filename=MODEL_FILE,
    revision=model_revision, local_dir=MODEL_DIR
))
print("Hugging Face model revision:", model_revision)
print("GGUF:", MODEL_PATH)
print("GGUF size (GiB):", round(MODEL_PATH.stat().st_size / (1024 ** 3), 2))

## 5. Запуск локального сервера с tool calling

Критичен флаг `--jinja`, а имя модели должно совпадать с `--alias qwen-3.5`. Сервер слушает только `127.0.0.1` внутри Colab. **Порт выбирается автоматически**, поэтому занятый `8080` не мешает запуску. Выбранный адрес используется для `/health`, проверки `tool_calls` и запуска CLI. Если запуск всё же завершится ошибкой, будет показан хвост лога.

In [ ]:
import socket
import time
from urllib.error import URLError
from urllib.request import urlopen

# A previous Colab cell may already have a server bound to port 8080.
# Stop only this notebook's earlier process; never kill unrelated servers.
previous_process = globals().get("SERVER_PROCESS")
if previous_process is not None and previous_process.poll() is None:
    previous_process.terminate()
    try:
        previous_process.wait(timeout=20)
    except subprocess.TimeoutExpired:
        previous_process.kill()
        previous_process.wait(timeout=10)
previous_log = globals().get("SERVER_LOG_HANDLE")
if previous_log is not None and not previous_log.closed:
    previous_log.close()

# Request a free loopback port from the OS, rather than assuming 8080.
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as port_probe:
    port_probe.bind(("127.0.0.1", 0))
    SERVER_PORT = port_probe.getsockname()[1]
SERVER_BASE_URL = f"http://127.0.0.1:{SERVER_PORT}/v1"
SERVER_HEALTH_URL = f"http://127.0.0.1:{SERVER_PORT}/health"
print("Selected free loopback port:", SERVER_PORT)
SERVER_LOG_PATH = WORKDIR / "llama-server.log"
SERVER_LOG_HANDLE = SERVER_LOG_PATH.open("w", encoding="utf-8")
SERVER_PROCESS = subprocess.Popen(
    [
        str(SERVER_BIN),
        "--model", str(MODEL_PATH),
        "--alias", MODEL_ALIAS,
        "--jinja",
        "--ctx-size", "8192",
        "--n-gpu-layers", "99",
        "--parallel", "1",
        "--flash-attn", "on",
        "--cache-type-k", "q8_0",
        "--cache-type-v", "q8_0",
        "--cache-prompt",
        "--reasoning", "off",
        "--timeout", "600",
        "--host", "127.0.0.1",
        "--port", str(SERVER_PORT),
    ],
    stdout=SERVER_LOG_HANDLE,
    stderr=subprocess.STDOUT,
    env=SERVER_ENV,
)
print("llama-server PID:", SERVER_PROCESS.pid)
ready = False
deadline = time.monotonic() + 180
while time.monotonic() < deadline:
    if SERVER_PROCESS.poll() is not None:
        break
    try:
        with urlopen(SERVER_HEALTH_URL, timeout=3) as response:
            if response.status == 200:
                ready = True
                break
    except (URLError, TimeoutError, OSError):
        pass
    time.sleep(2)

if not ready:
    SERVER_LOG_HANDLE.flush()
    tail = SERVER_LOG_PATH.read_text(errors="replace").splitlines()[-35:]
    print("Server log (last lines):\n" + "\n".join(tail))
    raise RuntimeError("llama-server failed to become healthy")
print("llama-server healthy on", SERVER_BASE_URL)

## 6. Диагностика tool calling (отдельно от агента)

Короткая проверка, возвращает ли сама Qwen структурированный `tool_calls` через наш `LocalOpenAIModel`. Если модель вместо инструмента пишет текст, это не считается пройденной проверкой — такой результат важно сохранить для отладки шаблона/версии llama.cpp.

In [ ]:
import sys as system_module

if str(REPO_DIR / "src") not in system_module.path:
    system_module.path.insert(0, str(REPO_DIR / "src"))

from github2cv_ml.agent.runner import LocalOpenAIModel

model = LocalOpenAIModel(
    base_url=SERVER_BASE_URL, model=MODEL_ALIAS, timeout=180,
    temperature=MODEL_TEMPERATURE
)
diagnostic_tool = [{
    "type": "function",
    "function": {
        "name": "echo_probe",
        "description": "Return the provided text without modification",
        "parameters": {
            "type": "object",
            "properties": {"text": {"type": "string"}},
            "required": ["text"],
        },
    },
}]
probe = model.complete(
    [
        {"role": "system", "content": "Call echo_probe once. Do not answer in prose."},
        {"role": "user", "content": "Call echo_probe with text ping."},
    ],
    tools=diagnostic_tool,
)
probe_calls = probe.get("tool_calls") if isinstance(probe, dict) else None
print("Structured tool call (required mode):", bool(probe_calls))
if probe_calls:
    print("Tool name:", probe_calls[0].get("function", {}).get("name"))
    print("Arguments:", probe_calls[0].get("function", {}).get("arguments"))
else:
    print("WARNING: Model did not produce tool_calls.")
    print("Response:", str(probe.get("content") if isinstance(probe, dict) else probe)[:800])
    print("The model did not call a tool even with tool_choice=required.")
    print("Check --jinja, chat template, and llama-server version before trusting agent output.")

## 7. Реальный end-to-end запуск Repo Analyzer

Вызываем **настоящую CLI-команду**, которая поставляется в PR #30. Изменив `TARGET_REPO`, можно тестировать другие публичные репозитории. Параметры бюджета контролируют число tool calls и количество прочитанных фрагментов. Если endpoint не работает, приложение должно сохранить неполный отчёт, а не потерять уже собранные данные.

In [ ]:
import json

cmd = [
    sys.executable, "-m", "github2cv_ml.agent", TARGET_REPO,
    "--base-url", SERVER_BASE_URL,
    "--model", MODEL_ALIAS,
    "--temperature", str(MODEL_TEMPERATURE),
    "--max-steps", str(MAX_STEPS),
    "--max-reads", str(MAX_READS),
    "--output", str(RESULT_DIR),
]
started = time.monotonic()
run = subprocess.run(
    cmd, cwd=str(REPO_DIR), env=os.environ.copy(),
    text=True, capture_output=True, timeout=2400,
)
elapsed = time.monotonic() - started
print("CLI exit code:", run.returncode)
print("Runtime (seconds):", round(elapsed, 1))
print("stdout:\n" + run.stdout[-4000:])
if run.stderr:
    print("stderr:\n" + run.stderr[-5000:])
if run.returncode != 0:
    raise RuntimeError("Repo Analyzer CLI failed; see stderr above")
if not (RESULT_DIR / "analysis.json").exists():
    raise RuntimeError("CLI finished but analysis.json is absent")
analysis = json.loads((RESULT_DIR / "analysis.json").read_text(encoding="utf-8"))
print("Run status:", analysis["status"])
print("Evidence count:", len(analysis["evidence"]))
print("Tool actions:", len(analysis["tool_trace"]))
print("Selected-file coverage:", analysis.get("coverage", {}).get("planned_coverage_ratio"))

## 8. Проверка результата и источников

Здесь видно, **что именно делала модель**: какие инструменты вызывала, сколько было успешных чтений и что извлекла. Открой ссылки на код и вручную проверь смысл первых 3–5 claims. `verification_level=locator_only` означает, что автоматической семантической проверки здесь нет.

In [ ]:
from IPython.display import Markdown, display

trace = analysis["tool_trace"]
read_ok = sum(
    action.get("tool") == "read_file" and action.get("status") == "ok"
    for action in trace
)
errors = sum(action.get("status") == "error" for action in trace)
completed = analysis["status"] == "completed"
has_evidence = bool(analysis["evidence"])
print("Repository:", analysis["repository"]["url"])
print("Pinned commit SHA:", analysis["commit_sha"])
print("Status:", analysis["status"])
print("Verification level:", analysis["verification_level"])
print("Read calls (successful):", read_ok)
print("Tool errors:", errors)
recoveries = [action for action in trace if action.get("origin") == "automatic_recovery"]
print("Automatic recoveries:", len(recoveries))
print("Warnings:", analysis["warnings"])
print("Source-map plan (filename hypotheses):")
for item in analysis.get("exploration_plan", []):
    print(" ", item["candidate_role"], item["path"])
print("Coverage of selected targets (not whole repository):")
print(json.dumps(analysis.get("coverage", {}), indent=2, ensure_ascii=False))
print()
print("Exploration trace:")
for action in trace:
    print(
        "step", action.get("step"), "|", action.get("tool"),
        "|", action.get("status"), "|", action.get("arguments", {})
    )

print("\nExtracted evidence:")
for evidence in analysis["evidence"][:10]:
    source = evidence["source"]
    print(
        f"- {evidence['observation']}\n"
        f"  {source['path']}:{source['line_start']}-{source['line_end']}\n"
        f"  {source['url']}"
    )
print("\nManual audit: verify the first 3–5 evidence URLs against the claims.")
if (completed or analysis["status"] == "read_budget_exhausted" or analysis["status"] == "plan_covered") and read_ok > 0 and has_evidence:
    print("SMOKE RESULT: agent used read_file and produced locator-backed claims.")
    if analysis["status"] == "read_budget_exhausted":
        print("Explorer stopped at its read budget (not an explicit finish_analysis).")
    elif analysis["status"] == "plan_covered":
        print("Heuristic source plan was visited; this is not whole-repository or semantic coverage.")
elif analysis["status"] == "stalled":
    print("SMOKE RESULT: stalled after repeated errors; partial evidence is retained.")
    print("Inspect the trace and unvisited planned files; not a completed run.")
else:
    print("SMOKE RESULT: no successful completed/limited exploration, or no observations.")
    print("Inspect llama-server.log and tool_trace before drawing conclusions.")
display(Markdown((RESULT_DIR / "report.md").read_text(encoding="utf-8")))

## 9. Живая генерация навыков и абзацев — именно LLM

Здесь запускается новый **Candidate Profile Builder**: он читает сохранённый `analysis.json`, вызывает локальную Qwen в режиме JSON, выбирает навыки и создаёт связный абзац для каждого проекта. Python проверяет ссылки на evidence и убирает дубли. **Нет предопределённого словаря навыков.**

Для демонстрации нескольких репозиториев можно загрузить другие `analysis.json` или ZIP в Colab и добавить их пути в `CACHED_ANALYSIS_FILES` в первой ячейке. Пример: `["/content/second_repo_smoke.zip"]`. Эти файлы представляют кэш бэкенда: Repo Analyzer для них повторно **не запускается**.

Сохраняются `candidate_profile.json`, `profile_preview.md` и `llm_profile_trace.json` (в последнем видно, какие именно навыки и описания предложила модель). Это демонстрация Profile Builder, не финальное CV и не семантическая проверка claims.

In [ ]:
PROFILE_DIR = WORKDIR / "candidate_profile"
if not analysis["evidence"]:
    print("В анализе нет evidence: LLM Profile Builder пока нечего обобщать.")
else:
    profile_cmd = [
        sys.executable, "-m", "github2cv_ml.profile_demo",
        "--analysis", str(RESULT_DIR / "analysis.json"),
        "--github-login", CANDIDATE_LOGIN,
        "--base-url", SERVER_BASE_URL,
        "--model", MODEL_ALIAS,
        "--temperature", str(MODEL_TEMPERATURE),
        "--output", str(PROFILE_DIR),
    ]
    for cached_artifact in CACHED_ANALYSIS_FILES:
        profile_cmd.extend(["--analysis", str(cached_artifact)])
    profile_run = subprocess.run(
        profile_cmd, cwd=str(REPO_DIR), env=os.environ.copy(),
        text=True, capture_output=True, timeout=1200,
    )
    print("Profile Builder CLI exit code:", profile_run.returncode)
    print(profile_run.stdout[-4000:])
    if profile_run.stderr:
        print(profile_run.stderr[-5000:])
    if profile_run.returncode:
        raise RuntimeError("Profile generation failed; see CLI output above")

    candidate_profile = json.loads(
        (PROFILE_DIR / "candidate_profile.json").read_text(encoding="utf-8")
    )
    llm_trace = json.loads(
        (PROFILE_DIR / "llm_profile_trace.json").read_text(encoding="utf-8")
    )
    skills = [
        claim["text"] for claim in candidate_profile["claims"]
        if claim["kind"] == "skill"
    ]
    print("Реальных вызовов LLM:", len(llm_trace))
    print("Найденные навыки:", ", ".join(skills))
    for call in llm_trace:
        print("LLM обработала:", call["repository"])
        print("  Количество evidence:", call["evidence_count"])
        print("  Сама предложила навыки:", call["llm_proposed_skills"])
        print("  Сама написала:", call["llm_proposed_project_paragraph"])

    display(Markdown(
        (PROFILE_DIR / "profile_preview.md").read_text(encoding="utf-8")
    ))

## 10. Скачать результаты двух ML-стадий

Скачиваем `analysis.json`, `report.md`, а если Profile Builder отработал — и его JSON, Markdown и LLM trace. Они могут включать извлечённые сведения о репозитории и пути к файлам; не публикуй артефакты анализа приватных проектов без проверки. Лог `llama-server` остаётся в Colab и доступен для отладки.

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

bundle_path = WORKDIR / "github2cv_live_profile_demo.zip"
with ZipFile(bundle_path, "w", compression=ZIP_DEFLATED) as bundle:
    for filename in ("analysis.json", "report.md"):
        result_file = RESULT_DIR / filename
        if result_file.exists():
            bundle.write(result_file, arcname=filename)
    for filename in ("candidate_profile.json", "profile_preview.md", "llm_profile_trace.json"):
        result_file = PROFILE_DIR / filename
        if result_file.exists():
            bundle.write(result_file, arcname=f"profile/{filename}")
print("Results archive:", bundle_path)
try:
    from google.colab import files

    files.download(str(bundle_path))
except ImportError:
    print("Not in Colab; open the ZIP at the location above.")

## 11. Остановить сервер (необязательно)

После проверки освобождает VRAM. Ячейку можно выполнять только после завершения анализа.

In [ ]:
if "SERVER_PROCESS" in globals() and SERVER_PROCESS.poll() is None:
    SERVER_PROCESS.terminate()
    try:
        SERVER_PROCESS.wait(timeout=20)
    except subprocess.TimeoutExpired:
        SERVER_PROCESS.kill()
        SERVER_PROCESS.wait(timeout=10)
if "SERVER_LOG_HANDLE" in globals():
    SERVER_LOG_HANDLE.close()
print("llama-server stopped (if it was running).")